# Ingesta

## import

In [58]:
from google.colab import auth, files
import gspread
from google.auth import default
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

## extraction

In [59]:

# 1. Autenticar el usuario (te pedirá permisos al ejecutar)
auth.authenticate_user()
# 2. Autorizar gspread
creds, _ = default()
gc = gspread.authorize(creds)
# 3. Abrir el spreadsheet por URL
spreadsheet_url = 'https://docs.google.com/spreadsheets/d/1josoma_4trLT7cSXwBlVsR1nRjBTzqVHthvjB_qTINE/edit?usp=sharing'
worksheet = gc.open_by_url(spreadsheet_url).sheet1
# 4. Obtener los datos y pasarlos a un DataFrame de pandas
rows = worksheet.get_all_values()
if rows:
    df = pd.DataFrame(rows[1:], columns=rows[0])
else:
    print('El spreadsheet está vacío.')

## transformaton

### filter rows

In [60]:
initial_rows = len(df)

# Valores permitidos para la columna 'edad'
edad_permitida = [
    "Entre 18 y 29 años",
    "Entre 30 y 39 años",
    "Entre 40 y 50 años"
]

# Filtrar el DataFrame
df_filtered = df[df['3. Edad:'].isin(edad_permitida)].copy()

# Contar filas eliminadas y restantes
removed_rows = initial_rows - len(df_filtered)
remaining_rows = len(df_filtered)

print(f"Filas iniciales: {initial_rows}")
print(f"Filas eliminadas: {removed_rows}")
print(f"Filas restantes después del filtro: {remaining_rows}")

df = df_filtered.copy()

Filas iniciales: 115
Filas eliminadas: 11
Filas restantes después del filtro: 104


### drop cols

In [61]:
if 'Columna 23' in df.columns:
    df = df.drop(columns=['Columna 23'])
    print("Columna 'Columna 23' eliminada exitosamente.")
else:
    print("La columna 'Columna 23' no se encontró en el DataFrame.")

Columna 'Columna 23' eliminada exitosamente.


### DOWNLOAD

In [62]:
df.to_csv('encuesta_raw.csv', index=False)
files.download('encuesta_raw.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### rename cols

In [63]:
def rename_columns(df, start_old_col, new_col_name):
  column_to_rename = None
  for col in df.columns:
      if col.startswith(start_old_col):
          column_to_rename = col
          break

  if column_to_rename:
      df = df.rename(columns={column_to_rename: new_col_name})
  else:
      pass

  return df

In [64]:
start_old_col = "Consentimiento informado"
new_col_name = "consentimiento_informado"
df = rename_columns(df, start_old_col, new_col_name)

start_old_col = "Ingresá el código de identificación"
new_col_name = "codigo_identificacion"
df = rename_columns(df, start_old_col, new_col_name)

start_old_col = "Marca temporal"
new_col_name = "marca_temporal"
df = rename_columns(df, start_old_col, new_col_name)

start_old_col = "1. Genero:"
new_col_name = "genero"
df = rename_columns(df, start_old_col, new_col_name)

start_old_col = "2. Nivel educativo alcanzado:"
new_col_name = "nivel_educativo"
df = rename_columns(df, start_old_col, new_col_name)

start_old_col = "3. Edad:"
new_col_name = "edad"
df = rename_columns(df, start_old_col, new_col_name)

start_old_col = "4."
new_col_name = "consumo_personal"
df = rename_columns(df, start_old_col, new_col_name)

start_old_col = " 5."
new_col_name = "consumo_familiar"
df = rename_columns(df, start_old_col, new_col_name)

start_old_col = "6.¿Tus estudios o tu profesión están vinculados al ámbito de la salud?"
new_col_name = "conocimientos_salud"
df = rename_columns(df, start_old_col, new_col_name)

In [65]:
preguntas_rename = {
    'pregunta_inicio': '7.¿Qué es lo primero que pensás cuando escuchas que alguien toma medicación psiquiátrica?  ',
    'creencia_1': '8. Indique su grado de acuerdo con la siguiente afirmación:\n  \n"Una persona que toma medicación psiquiátrica puede desempeñarse en actividades laborales."',
    'creencia_2': '9. Indique su grado de acuerdo con la siguiente afirmación:\n  \n"La medicación psiquiátrica puede contribuir al manejo de las dificultades de salud mental de una persona."',
    'actitud_1': '10. Indique su grado de acuerdo con la siguiente afirmación: \n\n “Me sentiría cómodo/a estando a solas con una persona que toma medicación psiquiátrica.” ',
    'actitud_2': '11. Indique su grado de acuerdo con la siguiente afirmación: \n\n “Me sentiría cómodo/a hablando con mis amigos o conocidos acerca de mi uso de medicación psiquiátrica.” ',
    'actitud_3': '12. Indique su grado de acuerdo con la siguiente afirmación:\n“Considero que Juan puede continuar desempeñando las tareas correspondientes a su puesto mientras realiza su tratamiento.” ',
    'creencia_3': '13. Indique su grado de acuerdo con la siguiente afirmación: \n\n “El hecho de que Juan tome medicación psiquiátrica influiría en mi percepción sobre su desempeño laboral.” ',
    'conducta_1': '14.  Indique su grado de acuerdo con la siguiente afirmación: \n\n  “Estaría dispuesto/a a realizar junto con Juan una tarea correspondiente al trabajo que comparten.”  ',
    'creencia_4': '15. Indique su grado de acuerdo con la siguiente afirmación: \n\n “Considero que una persona que toma medicación psiquiátrica puede desarrollar proyectos personales y familiares.”',
    'actitud_4': '16.  Indique su grado de acuerdo con la siguiente afirmación: \n\n  “El hecho de que María tome medicación psiquiátrica influiría en mi percepción sobre sus proyectos personales.”  ',
    'conducta_2': '17.  Indique su grado de acuerdo con la siguiente afirmación: \n\n“Si una persona cercana iniciara un tratamiento con medicación psiquiátrica, mantendría con ella las actividades que realizamos habitualmente.” ',
    'conducta_3': '18.  Indique su grado de acuerdo con la siguiente afirmación: \n\n“El hecho de saber que una persona cercana toma medicación psiquiátrica influiría en la manera en que me relaciono con ella.”',
    'conducta_4': '19.  Indique su grado de acuerdo con la siguiente afirmación: \n\n  “Si un profesional de la salud me indicara iniciar un tratamiento con medicación psiquiátrica, compartiría esa información con al menos una persona de confianza.”  ',
    'pregunta_final': '20. ¿Hay alguna experiencia, situación o información que haya influido en tu manera de pensar sobre las personas que toman medicación psiquiátrica? Si es así, contanos brevemente cuál.\nSi no tenés ninguna experiencia o situación que quieras mencionar, podés indicarlo.',
}

# Invertimos el diccionario para mapear de valor a clave
rename_mapping = {v: k for k, v in preguntas_rename.items()}

# Renombramos las columnas del DataFrame
df = df.rename(columns=rename_mapping)


### formateo valores

In [66]:
cols_to_convert = ['consumo_personal', 'consumo_familiar', 'conocimientos_salud']
for col in cols_to_convert:
    df[col] = df[col].replace({'Si': 1, 'No': 0}).astype(int)

/tmp/ipykernel_785/2364640681.py:3: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

/tmp/ipykernel_785/2364640681.py:3: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

/tmp/ipykernel_785/2364640681.py:3: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



In [67]:
cols = [
    'codigo_identificacion', 'creencia_1', 'creencia_2', 'actitud_1', 'actitud_2', 'actitud_3', 'creencia_3', 'conducta_1', 'creencia_4', 'actitud_4', 'conducta_2', 'conducta_3', 'conducta_4'
]

for col in cols:
    df[col] = df[col].str.strip().str.upper()


In [68]:
cols_encuesta = ['creencia_1', 'creencia_2', 'actitud_1', 'actitud_2', 'actitud_3', 'creencia_3', 'conducta_1', 'creencia_4', 'actitud_4', 'conducta_2', 'conducta_3', 'conducta_4']

mapping_dict = {
  'TOTALMENTE DE ACUERDO': 1,
  'DE ACUERDO': 0.5,
  'NI DE ACUERDO NI EN DESACUERDO': 0,
  'EN DESACUERDO': -0.5,
  'TOTALMENTE EN DESACUERDO': -1
}

df[cols_encuesta] = df[cols_encuesta].replace(mapping_dict).infer_objects(copy=False)


/tmp/ipykernel_785/3202606688.py:11: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



In [69]:
df_original = df.copy()

In [70]:
# FALTA VALENCIA
valencia = {
  "8.": 1,
  "9.": 1,
  "10.": 1,
  "11.": 1,
  "12.": 1,
  "13.": -1,
  "14.": 1,
  "15.": 1,
  "16.": -1,
  "17.": 1,
  "18.": -1,
  "19.": 1
}

# Cruzar keys de preguntas_rename y valencia para multiplicar en df
for col_name, pregunta_texto in preguntas_rename.items():
    for val_key, val_value in valencia.items():
        if pregunta_texto.startswith(val_key):
            if col_name in df.columns:
                df[col_name] = df[col_name] * val_value
            break

### new cols

In [71]:
suma_cols = (
    df['consumo_personal']
    # + df['consumo_familiar']
    + df['conocimientos_salud']
)

df['grupo_encuestado'] = np.where(suma_cols == 0, 'Externo al tema', 'Interno al tema')

In [72]:
# Generar columna 'creencia'
cols_creencia = [col for col in df.columns if col.startswith('creencia')]
df['creencia'] = df[cols_creencia].mean(axis=1)

# Generar columna 'actitud'
cols_actitud = [col for col in df.columns if col.startswith('actitud')]
df['actitud'] = df[cols_actitud].mean(axis=1)

# Generar columna 'conducta'
cols_conducta = [col for col in df.columns if col.startswith('conducta')]
df['conducta'] = df[cols_conducta].mean(axis=1)


## funciones para gráficos

In [73]:
def grafico_encuestados_agrupados(df):
  print(f"Total de encuestados: {len(df)}")

  conteo_grupos = df['grupo_encuestado'].value_counts()

  print(f"Total Interno al tema: {conteo_grupos.get('Interno al tema', 0)}")
  print(f"Total Externo al tema: {conteo_grupos.get('Externo al tema', 0)}")

  # 2. Preparar los datos agrupados
  df_grouped = df.groupby(['codigo_identificacion', 'grupo_encuestado']).size().reset_index(name='count')

  # Obtener categorías únicas
  codigos = sorted(df['codigo_identificacion'].unique())
  grupos = df_grouped['grupo_encuestado'].unique()

  # 3. Construir el gráfico interactivo con Plotly
  fig = go.Figure()

  # Agregar barras por cada grupo
  for grupo in grupos:
      df_sub = df_grouped[df_grouped['grupo_encuestado'] == grupo]
      # Mapear counts asegurando el orden correcto de los códigos de identificación
      counts = [df_sub[df_sub['codigo_identificacion'] == c]['count'].values[0] if c in df_sub['codigo_identificacion'].values else 0 for c in codigos]

      fig.add_trace(go.Bar(
          x=codigos,
          y=counts,
          name=grupo,
          text=[str(v) if v > 0 else '' for v in counts], # Mostrar valor dentro del segmento si es mayor a 0
          textposition='inside'
      ))

  # Calcular totales de cada barra para colocarlos en la parte superior
  totales = [df_grouped[df_grouped['codigo_identificacion'] == c]['count'].sum() for c in codigos]

  # Agregar etiquetas de totales arriba de las barras
  annotations = []
  for i, total in enumerate(totales):
      annotations.append(dict(
          x=codigos[i],
          y=total,
          text=f"<b>Total: {total}</b>",
          xanchor='center',
          yanchor='bottom',
          showarrow=False,
          font=dict(size=12, color='black')
      ))

  # Configurar el diseño para barras apiladas y agregar interactividad
  fig.update_layout(
      title='Distribución de Grupo Encuestado por Código de Identificación',
      xaxis_title='Código de Identificación',
      yaxis_title='Cantidad',
      barmode='stack',
      annotations=annotations,
      legend_title='Grupo Encuestado',
      template='plotly_white',
      hovermode='x'
  )

  fig.show()

In [74]:
def lista_preguntas_mc(preguntas_rename):
    preguntas_limpias = {}

    for key, value in preguntas_rename.items():
        if not key.startswith("pregunta"):
            # Eliminar todos los caracteres desde el inicio hasta el primer punto
            if '.' in value:
                texto = value.split('.', 1)[1]
            else:
                texto = value

            # Eliminar la frase indicada
            texto = texto.replace("Indique su grado de acuerdo con la siguiente afirmación:", "")

            # Eliminar caracteres específicos
            for char in ['\n', '"', '“', '”']:
                texto = texto.replace(char, "")

            # Eliminar espacios a los extremos
            texto = texto.strip()

            preguntas_limpias[key] = texto

    for key, value in sorted(preguntas_limpias.items()):
        print(f"- {key}: {value}")


In [75]:
def grafico_respuestas_agrupadas(df, group_cols):
  # Agrupar por 'grupo_encuestado' y calcular el promedio de los valores
  df_agrupado = df.groupby('grupo_encuestado')[group_cols].mean().reset_index()

  # Transformar el dataframe a formato largo (melt) para facilitar el gráfico
  df_melted = df_agrupado.melt(id_vars='grupo_encuestado', var_name='columna', value_name='promedio')

  # Crear el gráfico de barras agrupadas
  fig = px.bar(
      df_melted,
      x='columna',
      y='promedio',
      color='grupo_encuestado',
      barmode='group',
      title='Promedio de valores por pregunta y grupo encuestado',
      labels={'columna': 'Columnas', 'promedio': 'Promedio de valores', 'grupo_encuestado': 'Grupo'}
  )

  # Ordenar alfabéticamente el eje X
  fig.update_xaxes(categoryorder='category ascending')

  # Agregar anotación explicativa
  fig.update_layout(
      annotations=[
          dict(
              text="Nota: Los puntajes positivos denotan muestra de empatía, y puntajes negativos denotan una posición negativa sobre el tema.",
              showarrow=False,
              xref="paper", yref="paper",
              x=0.5, y=-0.25, # Ubicado centrado debajo del gráfico
              xanchor="center", yanchor="top",
              font=dict(size=12, color="gray")
          )
      ],
      margin=dict(b=100) # Agregar margen inferior para que el texto no se corte
  )

  fig.show()


In [76]:
def grafico_correlacion(df, group_cols, grupo=None):
  """
  Genera un mapa de calor con la matriz de correlación.
  La correlación (.corr) se calcula utilizando el método de Pearson sobre los
  registros individuales (cada fila) para las columnas especificadas,
  sin sumar ni promediar previamente.
  """
  # Filtrar por grupo si se proporciona
  if grupo:
      df_filtered = df[df['grupo_encuestado'] == grupo]
      title = f'Matriz de Correlación de las Respuestas - {grupo}'
  else:
      df_filtered = df
      title = 'Matriz de Correlación de las Respuestas'

  # Ordenar las columnas de la encuesta alfabéticamente
  group_cols_sorted = sorted(group_cols)

  # Calcular la matriz de correlación para las columnas de la encuesta
  df_corr = df_filtered[group_cols_sorted].corr()

  # Definir la escala de colores personalizada
  custom_colorscale = [
      [0.0, 'rgb(139,0,0)'],       # -1: Rojo oscuro
      [0.25, 'rgb(255,100,100)'],  # -0.5 (aprox en escala 0-1): Rojo claro
      [0.25, 'white'],             # Inicio de la zona blanca (-0.5)
      [0.75, 'white'],             # Fin de la zona blanca (0.5)
      [0.75, 'rgb(144,238,144)'],  # 0.5 (aprox en escala 0-1): Verde claro
      [1.0, 'rgb(0,100,0)']        # 1: Verde oscuro
  ]

  # Crear el mapa de calor (heatmap)
  fig = px.imshow(
      df_corr,
      text_auto='.2f',
      aspect='auto',
      color_continuous_scale=custom_colorscale,
      zmin=-1, # Asegurar que la escala mapee correctamente los valores de -1 a 1
      zmax=1,
      title=title
  )

  # Agregar anotación explicativa
  fig.update_layout(
      annotations=[
          dict(
              text="Nota: La correlación de Pearson se calcula comparando las respuestas individuales fila por fila,<br>no mediante la suma o promedio previo de los grupos.",
              showarrow=False,
              xref="paper", yref="paper",
              x=0.5, y=-0.25, # Ubicado centrado debajo del gráfico
              xanchor="center", yanchor="top",
              font=dict(size=12, color="gray")
          )
      ],
      margin=dict(b=100) # Agregar margen inferior para que el texto no se corte
  )

  fig.show()


# Exploración

In [84]:
df.to_csv('encuesta_clean.csv', index=False)
files.download('encuesta_clean.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [77]:
grafico_encuestados_agrupados(df)

Total de encuestados: 104
Total Interno al tema: 40
Total Externo al tema: 64


In [78]:
grafico_respuestas_agrupadas(df, ['creencia', 'actitud', 'conducta'])

In [79]:
grafico_respuestas_agrupadas(df, cols_encuesta)
print('')
lista_preguntas_mc(preguntas_rename)


- actitud_1: Me sentiría cómodo/a estando a solas con una persona que toma medicación psiquiátrica.
- actitud_2: Me sentiría cómodo/a hablando con mis amigos o conocidos acerca de mi uso de medicación psiquiátrica.
- actitud_3: Considero que Juan puede continuar desempeñando las tareas correspondientes a su puesto mientras realiza su tratamiento.
- actitud_4: El hecho de que María tome medicación psiquiátrica influiría en mi percepción sobre sus proyectos personales.
- conducta_1: Estaría dispuesto/a a realizar junto con Juan una tarea correspondiente al trabajo que comparten.
- conducta_2: Si una persona cercana iniciara un tratamiento con medicación psiquiátrica, mantendría con ella las actividades que realizamos habitualmente.
- conducta_3: El hecho de saber que una persona cercana toma medicación psiquiátrica influiría en la manera en que me relaciono con ella.
- conducta_4: Si un profesional de la salud me indicara iniciar un tratamiento con medicación psiquiátrica, compartiría e

In [80]:
grafico_correlacion(df, ['creencia', 'actitud', 'conducta'], 'Interno al tema')

In [81]:
grafico_correlacion(df, ['creencia', 'actitud', 'conducta'], 'Externo al tema')

In [82]:
grafico_correlacion(df, cols_encuesta, 'Interno al tema')
print('')
lista_preguntas_mc(preguntas_rename)


- actitud_1: Me sentiría cómodo/a estando a solas con una persona que toma medicación psiquiátrica.
- actitud_2: Me sentiría cómodo/a hablando con mis amigos o conocidos acerca de mi uso de medicación psiquiátrica.
- actitud_3: Considero que Juan puede continuar desempeñando las tareas correspondientes a su puesto mientras realiza su tratamiento.
- actitud_4: El hecho de que María tome medicación psiquiátrica influiría en mi percepción sobre sus proyectos personales.
- conducta_1: Estaría dispuesto/a a realizar junto con Juan una tarea correspondiente al trabajo que comparten.
- conducta_2: Si una persona cercana iniciara un tratamiento con medicación psiquiátrica, mantendría con ella las actividades que realizamos habitualmente.
- conducta_3: El hecho de saber que una persona cercana toma medicación psiquiátrica influiría en la manera en que me relaciono con ella.
- conducta_4: Si un profesional de la salud me indicara iniciar un tratamiento con medicación psiquiátrica, compartiría e

In [83]:
grafico_correlacion(df, cols_encuesta, 'Externo al tema')
print('')
lista_preguntas_mc(preguntas_rename)


- actitud_1: Me sentiría cómodo/a estando a solas con una persona que toma medicación psiquiátrica.
- actitud_2: Me sentiría cómodo/a hablando con mis amigos o conocidos acerca de mi uso de medicación psiquiátrica.
- actitud_3: Considero que Juan puede continuar desempeñando las tareas correspondientes a su puesto mientras realiza su tratamiento.
- actitud_4: El hecho de que María tome medicación psiquiátrica influiría en mi percepción sobre sus proyectos personales.
- conducta_1: Estaría dispuesto/a a realizar junto con Juan una tarea correspondiente al trabajo que comparten.
- conducta_2: Si una persona cercana iniciara un tratamiento con medicación psiquiátrica, mantendría con ella las actividades que realizamos habitualmente.
- conducta_3: El hecho de saber que una persona cercana toma medicación psiquiátrica influiría en la manera en que me relaciono con ella.
- conducta_4: Si un profesional de la salud me indicara iniciar un tratamiento con medicación psiquiátrica, compartiría e